In [ ]:
# Jersey Number Pipeline - Full setup and test (Kaggle)
# Uses original repo: https://github.com/mkoshkina/jersey-number-pipeline
# One cell: clone repos, download dataset & weights, run test pipeline

import os
import subprocess
import sys

WORK = "/kaggle/working" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else os.path.abspath(".")
REPO = "jersey-number-pipeline"
REPO_URL = "https://github.com/mkoshkina/jersey-number-pipeline"
REPO_ROOT = os.path.join(WORK, REPO)

def run(cmd, cwd=None, check=True):
    cwd = cwd or WORK
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and r.returncode != 0:
        raise SystemExit(r.returncode)
    return r

# 1. Clone main repo (original)
if not os.path.isdir(os.path.join(REPO_ROOT, ".git")):
    run(f'git clone --depth 1 "{REPO_URL}" "{REPO}"', cwd=WORK)
else:
    print("Repo already cloned.")

# 2. Clone sub-repos (README/setup.py)
os.makedirs(REPO_ROOT, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_ROOT, "sam2")):
    run("git clone --recurse-submodules --depth 1 https://github.com/davda54/sam.git sam2", cwd=REPO_ROOT)
os.makedirs(os.path.join(REPO_ROOT, "reid"), exist_ok=True)
if not os.path.isdir(os.path.join(REPO_ROOT, "reid", "centroids-reid")):
    run("git clone --recurse-submodules --depth 1 https://github.com/mikwieczorek/centroids-reid.git reid/centroids-reid", cwd=REPO_ROOT)
os.makedirs(os.path.join(REPO_ROOT, "reid", "centroids-reid", "models"), exist_ok=True)
os.makedirs(os.path.join(REPO_ROOT, "pose"), exist_ok=True)
if not os.path.isdir(os.path.join(REPO_ROOT, "pose", "ViTPose")):
    run("git clone --recurse-submodules --depth 1 https://github.com/ViTAE-Transformer/ViTPose.git pose/ViTPose", cwd=REPO_ROOT)
os.makedirs(os.path.join(REPO_ROOT, "str"), exist_ok=True)
if not os.path.isdir(os.path.join(REPO_ROOT, "str", "parseq")):
    run("git clone --recurse-submodules --depth 1 https://github.com/baudm/parseq.git str/parseq", cwd=REPO_ROOT)
print("All repos cloned.")

# 3. Pip install (main deps + sub-repo deps for no-conda run)
run(
    "pip install -q gdown SoccerNet torch torchvision opencv-python Pillow numpy pandas scipy tqdm pytorch-lightning yacs",
    cwd=REPO_ROOT,
)
if os.path.isfile(os.path.join(REPO_ROOT, "reid", "centroids-reid", "requirements.txt")):
    run("pip install -q -r requirements.txt", cwd=os.path.join(REPO_ROOT, "reid", "centroids-reid"))
run("pip install -q -e .", cwd=os.path.join(REPO_ROOT, "str", "parseq"))

# 4. Download SoccerNet jersey-2023 dataset (README: SoccerNet/sn-jersey)
data_sn = os.path.join(REPO_ROOT, "data", "SoccerNet")
os.makedirs(data_sn, exist_ok=True)
jersey_dir = os.path.join(data_sn, "jersey-2023")
if not os.path.isdir(jersey_dir):
    exec(
        """
from SoccerNet.Downloader import SoccerNetDownloader
d = SoccerNetDownloader(LocalDirectory=%r)
d.downloadDataTask(task="jersey-2023", split=["train", "test"])
"""
        % data_sn
    )
    # Downloader may create data/SoccerNet/jersey-2023 or put train/test at top level
    if os.path.isdir(os.path.join(data_sn, "train")) and not os.path.isdir(jersey_dir):
        run(f'mkdir -p "{jersey_dir}"', cwd=REPO_ROOT)
        run(f'mv "{os.path.join(data_sn, "train")}" "{jersey_dir}/"', cwd=REPO_ROOT)
        run(f'mv "{os.path.join(data_sn, "test")}" "{jersey_dir}/"', cwd=REPO_ROOT)
    print("Dataset downloaded.")
else:
    print("Dataset already present.")

# 5. Download model weights (README + configuration.py)
os.makedirs(os.path.join(REPO_ROOT, "models"), exist_ok=True)
os.makedirs(os.path.join(REPO_ROOT, "pose", "ViTPose", "checkpoints"), exist_ok=True)
reid_models = os.path.join(REPO_ROOT, "reid", "centroids-reid", "models")
os.makedirs(reid_models, exist_ok=True)

# Legibility (SoccerNet), STR (SoccerNet), ViTPose, Centroid-ReID (2 ckpts)
weights = [
    ("18HAuZbge3z8TSfRiX_FzsnKgiBs-RRNw", os.path.join(REPO_ROOT, "models", "legibility_resnet34_soccer_20240215.pth")),
    ("1uRln22tlhneVt3P6MePmVxBWSLMsL3bm", os.path.join(REPO_ROOT, "models", "parseq_epoch=24-step=2575-val_accuracy=95.6044-val_NED=96.3255.ckpt")),
    ("1A3ftF118IcxMn_QONndR-8dPWpf7XzdV", os.path.join(REPO_ROOT, "pose", "ViTPose", "checkpoints", "vitpose-h.pth")),
    ("1w9yzdP_5oJppGIM4gs3cETyLujanoHK8", os.path.join(reid_models, "dukemtmcreid_resnet50_256_128_epoch_120.ckpt")),
    ("1ZFywKEytpyNocUQd2APh2XqTe8X0HMom", os.path.join(reid_models, "market1501_resnet50_256_128_epoch_120.ckpt")),
]
import gdown
for fid, path in weights:
    if not os.path.isfile(path):
        gdown.download(f"https://drive.google.com/uc?id={fid}", path, quiet=False)
print("Weights downloaded.")

# 6. Run test pipeline (README: python3 main.py SoccerNet test)
centroids_reid_root = os.path.join(REPO_ROOT, "reid", "centroids-reid")
sam2_root = os.path.join(REPO_ROOT, "sam2")
env = os.environ.copy()
env["PYTHONPATH"] = f"{sam2_root}:{centroids_reid_root}:{env.get('PYTHONPATH', '')}"
cmd = [sys.executable, "main.py", "SoccerNet", "test"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, env=env)
if result.returncode != 0:
    raise SystemExit(result.returncode)
print("Pipeline finished. Outputs in out/SoccerNetResults/")
# Note: If the run fails at 'Detecting pose', install ViTPose deps (mmcv, mmpose) per README.